# Resonance Lattice — Fabric build & refresh

Builds a `.rlat` knowledge model from files in `Files/<SOURCE_DIR>/`. First run encodes everything; later runs re-encode only what changed.

Writes to `Files/rlat/<KM_NAME>.rlat`. The search UDF reloads on the next call.

## Session size

`vCores: 8` cuts encode time about 4× vs the Fabric default of 2. For larger first builds bump to 16 or 32.

Session spinup is ~2 minutes regardless of vCore count. **For light refresh runs where little or nothing re-encodes, skip the cell below entirely** — the spinup cost dwarfs the build.

In [ ]:
%%configure -f
{
    "vCores": 8  // recommended: 4, 8, 16, 32, 64
}


In [ ]:
import os

# Cache the 600 MB encoder under OneLake so it survives session teardown.
os.environ["XDG_CACHE_HOME"] = "/lakehouse/default/Files/.rlat-cache"

%pip install -q "rlat[ann]>=2.1.0a15"

from resonance_lattice.fabric.hf_loader import fetch_encoder_from_hf
from resonance_lattice.install.encoder import PINNED_REVISION
fetch_encoder_from_hf(PINNED_REVISION)
print("encoder ready")


## Parameters

Set `SOURCE_DIR` to the folder under `Files/` you want indexed. `KM_NAME` becomes the output basename — the result lands at `Files/rlat/<KM_NAME>.rlat`.

In [ ]:
SOURCE_DIR = "docs"
KM_NAME = "team-docs"


In [ ]:
from pathlib import Path

import notebookutils

LAKEHOUSE = notebookutils.lakehouse.getWithProperties(
    notebookutils.runtime.context["defaultLakehouseName"]
)

# Fabric mounts the default lakehouse at /lakehouse/default/. local-mode
# .rlats record source paths under this prefix; the search UDF's
# OneLakeStore parses them back into Files/-relative keys at query time.
LOCAL_SOURCE = Path(f"/lakehouse/default/Files/{SOURCE_DIR}")
LOCAL_RLAT   = Path(f"/tmp/{KM_NAME}.rlat")
ONELAKE_RLAT = f"Files/rlat/{KM_NAME}.rlat"

if not LOCAL_SOURCE.exists():
    raise FileNotFoundError(
        f"{LOCAL_SOURCE} not found. Check SOURCE_DIR; the default lakehouse "
        f"is {LAKEHOUSE['displayName']!r}."
    )
print(f"source: {LOCAL_SOURCE} -> output: {ONELAKE_RLAT}")


## Build or refresh

If a `.rlat` already exists at the output path, download it and run `refresh_rlat` — only changed files get re-encoded. Otherwise do a full build in `local` mode so the archive stores source paths rather than source bytes.

In [ ]:
from resonance_lattice import (
    FilesystemSourceWalker,
    StoreMode,
    build_rlat,
    refresh_rlat,
)


def _download_existing(onelake_path, dst):
    abfss = f"{LAKEHOUSE['properties']['abfsPath']}/{onelake_path}"
    notebookutils.fs.cp(abfss, f"file://{dst}", recurse=False)


walker = FilesystemSourceWalker([LOCAL_SOURCE], LOCAL_SOURCE)

existing = notebookutils.fs.exists(
    f"{LAKEHOUSE['properties']['abfsPath']}/{ONELAKE_RLAT}"
)
if existing:
    print(f"[refresh] downloading existing {ONELAKE_RLAT} -> {LOCAL_RLAT}")
    if LOCAL_RLAT.exists():
        LOCAL_RLAT.unlink()
    _download_existing(ONELAKE_RLAT, LOCAL_RLAT)
    result = refresh_rlat(walker, LOCAL_RLAT, on_progress=print)
    print(
        f"[refresh] added={result.n_added} changed={result.n_changed} "
        f"deleted={result.n_deleted} unchanged={result.n_unchanged} "
        f"elapsed={result.elapsed_seconds:.1f}s"
    )
else:
    print(f"[build] no existing .rlat; full build from {LOCAL_SOURCE}")
    result = build_rlat(
        walker,
        LOCAL_RLAT,
        store_mode=StoreMode.LOCAL,
        on_progress=print,
    )
    print(
        f"[build] wrote {result.output_path} "
        f"({result.n_passages} passages from {result.n_files} files, "
        f"elapsed={result.elapsed_seconds:.1f}s)"
    )


## Upload

Copy the new `.rlat` back to OneLake. The search UDF's mtime cache picks up the change on the next call; no UDF redeploy needed.

In [ ]:
abfss_target = f"{LAKEHOUSE['properties']['abfsPath']}/{ONELAKE_RLAT}"
notebookutils.fs.cp(f"file://{LOCAL_RLAT}", abfss_target, recurse=False)
print(f"uploaded -> {ONELAKE_RLAT}")


## Telemetry

Write one JSON record per run to `Files/.rlat-builds/`. The analytics notebook (`fabric_analytics.ipynb`) consolidates these into the `udf_builds` Delta table.

In [ ]:
import json
import secrets
from datetime import datetime, timezone

now = datetime.now(timezone.utc)
ts_field = now.strftime("%Y-%m-%dT%H:%M:%S.%fZ")
ts_filename = now.strftime("%Y-%m-%dT%H-%M-%S")
telemetry_name = f"{ts_filename}.{secrets.token_hex(3)}.json"

row = {
    "ts": ts_field,
    "action": "refresh" if hasattr(result, "n_added") else "build",
    "km_name": KM_NAME,
    "source_dir": SOURCE_DIR,
    "n_passages": result.n_passages,
    "n_files": getattr(result, "n_files", None),
    "elapsed_seconds": result.elapsed_seconds,
    "encoder_revision": result.encoder_revision,
    "store_mode": getattr(result, "store_mode", "local"),
    "n_added": getattr(result, "n_added", None),
    "n_changed": getattr(result, "n_changed", None),
    "n_deleted": getattr(result, "n_deleted", None),
    "n_unchanged": getattr(result, "n_unchanged", None),
}

local_telemetry = Path(f"/tmp/{telemetry_name}")
local_telemetry.write_text(json.dumps(row), encoding="utf-8")
notebookutils.fs.cp(
    f"file://{local_telemetry}",
    f"{LAKEHOUSE['properties']['abfsPath']}/Files/.rlat-builds/{telemetry_name}",
    recurse=False,
)
print(f"telemetry -> .rlat-builds/{telemetry_name}")


## Schedule

Run this notebook from a Fabric Pipeline on a schedule if required. A refresh with no changed files is cheap.

For first builds over ~10k passages, Fabric CPU is slow. Build on a GPU host instead (Kaggle, Colab, local), upload the seed `.rlat` to `Files/rlat/`, and let scheduled refreshes take it from there.